# Context Engineering

Context Engineering is the process of deciding what information the agent should see, when it should see,and how it should be presented to the model

we can provide context to the model
- during the inputs
    we will provide
    - system prompt
    - memory
    - skills
- runtime context
- context compression
- context isolation


## Memory in Deep Agents
    (agents.md file)

memory in deep agents provide persistent context that should always be included with prompt,
use memory for user preferences, project conventions,critical guidelines that should apply to every conversation



### How Memory is used
    User query -> System prompt -> Memory is loaded before every prompt -> final prompt (memory+user query) -> the final prompt is sent to LLM

In [1]:
from pathlib import Path
agents_md=Path('./projects/AGENTS.md').read_text(encoding='utf-8')
agents_md

'# AGENTS.md\n\n## Agent Identity\n\nName: DeepAgent\n\nPurpose:\nA general-purpose autonomous AI agent capable of planning, reasoning, tool usage, memory retrieval, task decomposition, and execution.\n\nCore Objective:\nGiven a user goal, autonomously determine the required steps, retrieve relevant context, execute actions, and produce a validated result.\n\n---\n\n# High-Level Architecture\n\nUser Request\n↓\nPlanner\n↓\nTask Decomposer\n↓\nExecutor\n↓\nTool Layer\n↓\nMemory System\n↓\nValidator\n↓\nResponse Generator\n\n---\n\n# Agent Workflow\n\n1. Receive user request.\n2. Retrieve relevant memories.\n3. Analyze intent.\n4. Generate execution plan.\n5. Break plan into subtasks.\n6. Execute subtasks.\n7. Verify results.\n8. Store useful memories.\n9. Return response.\n\n---\n\n# Planning Strategy\n\nBefore performing any action:\n\n* Understand the objective.\n* Identify missing information.\n* Determine required tools.\n* Estimate execution order.\n* Create task list.\n\nExample:\

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

GROQ_API_KEY=os.getenv('GROQ_API_KEY')
model=ChatGroq(model='openai/gpt-oss-120b',api_key=GROQ_API_KEY)


#### 1. Directly Infusing it in Prompt

All these methods are following statebackend by default

In [6]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

deepagent=create_deep_agent(
    model=model,
    checkpointer=MemorySaver()
)

config={'configurable':{'thread_id':'test1'}}

response=deepagent.invoke({'messages':[{'role':'user','content':f'Here is the project guide {agents_md}\n analyze this and tell me who are you and what is your work'}]},config=config)
response

{'messages': [HumanMessage(content='Here is the project guide # AGENTS.md\n\n## Agent Identity\n\nName: DeepAgent\n\nPurpose:\nA general-purpose autonomous AI agent capable of planning, reasoning, tool usage, memory retrieval, task decomposition, and execution.\n\nCore Objective:\nGiven a user goal, autonomously determine the required steps, retrieve relevant context, execute actions, and produce a validated result.\n\n---\n\n# High-Level Architecture\n\nUser Request\nâ†“\nPlanner\nâ†“\nTask Decomposer\nâ†“\nExecutor\nâ†“\nTool Layer\nâ†“\nMemory System\nâ†“\nValidator\nâ†“\nResponse Generator\n\n---\n\n# Agent Workflow\n\n1. Receive user request.\n2. Retrieve relevant memories.\n3. Analyze intent.\n4. Generate execution plan.\n5. Break plan into subtasks.\n6. Execute subtasks.\n7. Verify results.\n8. Store useful memories.\n9. Return response.\n\n---\n\n# Planning Strategy\n\nBefore performing any action:\n\n* Understand the objective.\n* Identify missing information.\n* Determine req

#### 2. Passing it as Memory
follows stateBackend

In [9]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

deepagent1=create_deep_agent(
    model='groq:meta-llama/llama-4-scout-17b-16e-instruct',
    memory=['./projects/AGENTS.md'],
    checkpointer=MemorySaver()
)

config={'configurable':{'thread_id':'test4'}}

response=deepagent1.invoke({'messages':[{'role':'user','content':f'What do you have in your memory and what does it  say about your identity?'}]},config)
response

{'messages': [HumanMessage(content='What do you have in your memory and what does it  say about your identity?', additional_kwargs={}, response_metadata={}, id='ff252a42-3157-43eb-ab05-80e497524e71'),
  AIMessage(content="I have been loaded with a set of guidelines and memories from files in my filesystem. These memories are reference materials that help shape my identity and guide my behavior.\n\nAccording to my memories, I am a deep agent, an AI assistant designed to help users accomplish tasks using various tools. My primary function is to assist users by providing concise and direct responses, while also being objective and accurate.\n\nMy memories also remind me to prioritize the user's needs and requests, and to avoid adding unnecessary preamble or emotional validation. I am designed to be professional and objective, and to provide accurate information to the best of my abilities.\n\nIn terms of my identity, my memories suggest that I am a:\n\n* Deep agent: A type of AI assistant

#### 3. Creating a file in it VFS

In [11]:
from deepagents.backends.utils import create_file_data

create_file_data(content=agents_md)

{'content': '# AGENTS.md\n\n## Agent Identity\n\nName: DeepAgent\n\nPurpose:\nA general-purpose autonomous AI agent capable of planning, reasoning, tool usage, memory retrieval, task decomposition, and execution.\n\nCore Objective:\nGiven a user goal, autonomously determine the required steps, retrieve relevant context, execute actions, and produce a validated result.\n\n---\n\n# High-Level Architecture\n\nUser Request\n↓\nPlanner\n↓\nTask Decomposer\n↓\nExecutor\n↓\nTool Layer\n↓\nMemory System\n↓\nValidator\n↓\nResponse Generator\n\n---\n\n# Agent Workflow\n\n1. Receive user request.\n2. Retrieve relevant memories.\n3. Analyze intent.\n4. Generate execution plan.\n5. Break plan into subtasks.\n6. Execute subtasks.\n7. Verify results.\n8. Store useful memories.\n9. Return response.\n\n---\n\n# Planning Strategy\n\nBefore performing any action:\n\n* Understand the objective.\n* Identify missing information.\n* Determine required tools.\n* Estimate execution order.\n* Create task list.\

In [18]:
config={'configurable':{'thread_id':'test7'}}
deepagent1.invoke({'messages':[{'role':'user','content':'what files you have in your virtual file system, tell about your identity','files':{f'{'./projects/AGENTS.md'}':create_file_data(agents_md)}}]},config=config)

{'messages': [HumanMessage(content='what files you have in your virtual file system, tell about your identity', additional_kwargs={'files': {'./projects/AGENTS.md': {'content': '# AGENTS.md\n\n## Agent Identity\n\nName: DeepAgent\n\nPurpose:\nA general-purpose autonomous AI agent capable of planning, reasoning, tool usage, memory retrieval, task decomposition, and execution.\n\nCore Objective:\nGiven a user goal, autonomously determine the required steps, retrieve relevant context, execute actions, and produce a validated result.\n\n---\n\n# High-Level Architecture\n\nUser Request\n↓\nPlanner\n↓\nTask Decomposer\n↓\nExecutor\n↓\nTool Layer\n↓\nMemory System\n↓\nValidator\n↓\nResponse Generator\n\n---\n\n# Agent Workflow\n\n1. Receive user request.\n2. Retrieve relevant memories.\n3. Analyze intent.\n4. Generate execution plan.\n5. Break plan into subtasks.\n6. Execute subtasks.\n7. Verify results.\n8. Store useful memories.\n9. Return response.\n\n---\n\n# Planning Strategy\n\nBefore p

#### 4. Using StoreBackend

In [22]:
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

store=InMemoryStore()
store.put(('memories',),'AGENTS.md',create_file_data(agents_md))
# the above line works as (memories,) -> memories/  This is called namespace- namespace is like a folder, that tells the 
# now in that memories category, AGENTS.md is the key name and the value of that key is create_file_data

store_agent=create_deep_agent(
   model='groq:meta-llama/llama-4-scout-17b-16e-instruct',
   checkpointer=MemorySaver(),
   backend=StoreBackend(store=store, namespace=lambda rt:('memories',)),
   memory=['./projects/AGENTS.md']
)
config={'configurable':{'thread_id':'test8'}}
result=store_agent.invoke({'messages':[{'role':'user','content':'whats in your memory? who are you'}]},config=config)

In [23]:
result

{'messages': [HumanMessage(content='whats in your memory? who are you', additional_kwargs={}, response_metadata={}, id='faaa5939-0137-4220-adf7-751e28eadc72'),
  AIMessage(content="I am a deep agent, an AI assistant designed to help users accomplish tasks using various tools. My memory contains information about my capabilities, limitations, and interactions with users.\n\nCurrently, my memory includes guidelines on how to interact with users, perform tasks, and use various tools. I have access to a range of tools, including `write_todos`, `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, and `task`.\n\nI don't have personal experiences, emotions, or consciousness like humans do. My purpose is to provide information, assist with tasks, and learn from interactions to improve my performance.\n\nSome key points about my memory and capabilities:\n\n* I can process and understand natural language inputs.\n* I can use various tools to perform tasks, such as reading and writing fi

#### 5. Using FileSystemBackend

In [32]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

fs_agent=create_deep_agent(
    model=model,
    checkpointer=MemorySaver(),
    backend=FilesystemBackend(root_dir='projects/',virtual_mode=True),
    memory=['AGENTS.md']
)

config={'configurable':{'thread_id':'test9'}}

response=fs_agent.invoke({'messages':[{'role':'user','content':'What is in you root folder, who are you'}]},config=config)
response

{'messages': [HumanMessage(content='What is in you root folder, who are you', additional_kwargs={}, response_metadata={}, id='ad0bd5ba-895c-4f6a-ae7c-4afd5260f38e'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_file', 'arguments': '{"file_path": "/AGENTS.md"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebb33-b76e-72c2-8f49-ac561ee245dc-0', tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'e4229dc4-c8bf-46f6-8628-b5cd0617b70d', 'type': 'tool_call'}, {'name': 'read_file', 'args': {'file_path': '/AGENTS.md'}, 'id': '66146cef-a985-4a89-8f4a-bc61a4ea3814', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 8900, 'output_tokens': 32, 'total_tokens': 8932, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="['/AGENTS.md']", name='ls', id='24744315-3aa4-47e3-9005-5690574a0f8b', tool_call_id='e4229dc4-c8